In [6]:
import numpy as np
import pandas as pd
save_dir = "/scratch/user/s4634945/group_scratch/Projects_scripts/Self/CCI_Review/"

#### HUGO genes

In [11]:
# import requests
# import pandas as pd

# url = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
# headers = {"Accept": "text/tab-separated-values"}

# r = requests.get(url, headers=headers)
# with open("hgnc_genes.tsv", "wb") as f:
#     f.write(r.content)

HUGO_genes = pd.read_csv("hgnc_genes.tsv", sep="\t")

/scratch/temp/20989961/ipykernel_2553469/621998700.py:11: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  HUGO_genes = pd.read_csv("hgnc_genes.tsv", sep="\t")


### Data for Mod.WGCNA

In [5]:
def read_gct_as_df(filepath: str) -> pd.DataFrame:
    """
    Reads a .gct file into a pandas DataFrame.

    Parameters:
    filepath: str
        Path to the .gct file

    Returns:
    pd.DataFrame
        DataFrame with gene names as index and expression values for each sample
    """
    with open(filepath, 'r') as f:
        version = f.readline().strip()     # Skip version line
        dims = f.readline().strip().split()  # Read dimensions
        col_names = f.readline().strip().split('\t')  # Read column names

    df = pd.read_csv(filepath, sep='\t', skiprows=2)
    df = df.set_index('Description')  # Use gene names as index
    df.drop(columns=['id','Name'],inplace=True)
    return df
    

In [14]:
brain_hypothalamus = read_gct_as_df("/QRISdata/Q5058/IOC/Benchmark_datasets/WGCNA_GTex_RNA/gene_tpm_2017-06-05_v8_brain_hypothalamus.gct")
brain_hypothalamus = brain_hypothalamus.T
brain_hypothalamus = brain_hypothalamus.loc[:,brain_hypothalamus.columns.isin(HUGO_genes['symbol'])]
liver = read_gct_as_df("/QRISdata/Q5058/IOC/Benchmark_datasets/WGCNA_GTex_RNA/gene_tpm_2017-06-05_v8_liver.gct")
liver = liver.T
liver = liver.loc[:,liver.columns.isin(HUGO_genes['symbol'])]

In [15]:
attributes = pd.read_csv("/QRISdata/Q5058/IOC/Benchmark_datasets/WGCNA_GTex_RNA/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",sep="\t",index_col=0)
phenotype = pd.read_csv("/QRISdata/Q5058/IOC/Benchmark_datasets/WGCNA_GTex_RNA/GTEx_Analysis_v8_Annotations_SubjectPhenotypesDS.txt",sep="\t")

# Filtering based on RIN
attributes = attributes[attributes['SMRIN']>5.7]
# Filtering based on DTHHRDY (keep 1,2)
phenotype = phenotype[phenotype['DTHHRDY'].isin([1,2])]

attributes['SAMPID'] = attributes.index
attributes['SUBJID'] = attributes.index.str.split("-").str[:2].str.join("-")
attributes.set_index('SUBJID',inplace=True)

phenotype_attribute = pd.merge(phenotype,attributes[['SAMPID','SMTSISCH']],left_on="SUBJID",right_on="SUBJID",how="inner")
phenotype_attribute = phenotype_attribute[['SAMPID','SEX','AGE','DTHHRDY','SMTSISCH']]
phenotype_attribute['SAMPID_2'] = phenotype_attribute['SAMPID'].str.split('-').str[:2].str.join('-')

phenotype_attribute_brain_hypothalamus = phenotype_attribute[phenotype_attribute['SAMPID'].isin(brain_hypothalamus.index)]
phenotype_attribute_liver = phenotype_attribute[phenotype_attribute['SAMPID'].isin(liver.index)]
common_samples = list(set(phenotype_attribute_liver['SAMPID_2']).intersection(phenotype_attribute_brain_hypothalamus['SAMPID_2']))

common_samples_liver = liver[liver.index.str.split('-').str[:2].str.join('-').isin(common_samples)]
common_samples_brain_hypothalamus = brain_hypothalamus[brain_hypothalamus.index.str.split('-').str[:2].str.join('-').isin(common_samples)]

phenotype_attribute_brain_hypothalamus.set_index('SAMPID',inplace=True)
phenotype_attribute_liver.set_index('SAMPID',inplace=True)

aged_samples = phenotype_attribute_brain_hypothalamus[phenotype_attribute_brain_hypothalamus['AGE'].isin(["60-69","70-79"])].index.str.split('-').str[:2].str.join('-')
young_samples = phenotype_attribute_brain_hypothalamus[~phenotype_attribute_brain_hypothalamus['AGE'].isin(["60-69","70-79"])].index.str.split('-').str[:2].str.join('-')
aged_samples = set(aged_samples).intersection(common_samples)
young_samples = set(young_samples).intersection(common_samples)

young_phenotype_attribute_brain_hypothalamus = phenotype_attribute_brain_hypothalamus[phenotype_attribute_brain_hypothalamus['SAMPID_2'].isin(young_samples)]
young_phenotype_attribute_liver = phenotype_attribute_liver[phenotype_attribute_liver['SAMPID_2'].isin(young_samples)]

aged_phenotype_attribute_brain_hypothalamus = phenotype_attribute_brain_hypothalamus[phenotype_attribute_brain_hypothalamus['SAMPID_2'].isin(aged_samples)]
aged_phenotype_attribute_liver = phenotype_attribute_liver[phenotype_attribute_liver['SAMPID_2'].isin(aged_samples)]

In [16]:
import pickle

expr_dict = {
    "young_liver": liver[liver.index.isin(young_phenotype_attribute_liver.index)],
    "young_brain_hypothalamus": brain_hypothalamus[
        brain_hypothalamus.index.isin(young_phenotype_attribute_brain_hypothalamus.index)],
    "aged_liver": liver[liver.index.isin(aged_phenotype_attribute_liver.index)],
    "aged_brain_hypothalamus": brain_hypothalamus[
        brain_hypothalamus.index.isin(aged_phenotype_attribute_brain_hypothalamus.index)]}

with open("ModWGCNA_gene_expression_matrices.pkl", "wb") as f:
    pickle.dump(expr_dict, f)

pheno_dict = {
    "young_liver": young_phenotype_attribute_liver,
    "young_brain_hypothalamus": young_phenotype_attribute_brain_hypothalamus,
    "aged_liver": aged_phenotype_attribute_liver,
    "aged_brain_hypothalamus": aged_phenotype_attribute_brain_hypothalamus}

with open("ModWGCNA_phenotype_tables.pkl", "wb") as f:
    pickle.dump(pheno_dict, f)    

for name, df in pheno_dict.items():
    filename = f"{name}.csv"
    df.to_csv(filename, index=True)  # keep rownames/index
    print(f"Saved {filename}")

Saved young_liver.csv
Saved young_brain_hypothalamus.csv
Saved aged_liver.csv
Saved aged_brain_hypothalamus.csv


### Data for MultiCens

In [1]:
read_gct_as_df <- function(filepath) {
  # Read the file, skipping the first two lines (version + dimensions)
  df <- read.table(
    filepath,
    sep = "\t",
    header = TRUE,
    skip = 2,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )
  
  # Keep only the first occurrence if Description is duplicated
  df <- df[!duplicated(df$Description), ]
  
  # Set gene names as rownames
  rownames(df) <- df$Description
  
  # Drop unnecessary columns
  df <- df[, !colnames(df) %in% c("id", "Name", "Description")]
  
  return(df)}

In [2]:
HUGO_genes = read.csv("hgnc_genes.tsv", sep="\t")

In [13]:
brain_hypothalamus = read_gct_as_df("/QRISdata/Q5058/IOC/Benchmark_datasets/WGCNA_GTex_RNA/gene_reads_2017-06-05_v8_brain_hypothalamus.gct")
brain_hypothalamus = brain_hypothalamus[rownames(brain_hypothalamus) %in% HUGO_genes$symbol,]

liver = read_gct_as_df("/QRISdata/Q5058/IOC/Benchmark_datasets/WGCNA_GTex_RNA/gene_reads_2017-06-05_v8_liver.gct")
liver = liver[rownames(liver) %in% HUGO_genes$symbol,]

In [14]:
library(edgeR)
brain_hypothalamus_dge <- DGEList(brain_hypothalamus)
brain_hypothalamus_dge <- calcNormFactors(brain_hypothalamus_dge, method = "TMM")
brain_hypothalamus_norm_counts <- cpm(brain_hypothalamus_dge, normalized.lib.sizes = TRUE)
brain_hypothalamus_norm_counts <- t(brain_hypothalamus_norm_counts)

Loading required package: limma



In [15]:
liver_dge <- DGEList(liver)
liver_dge <- calcNormFactors(liver_dge, method = "TMM")
liver_norm_counts <- cpm(liver_dge, normalized.lib.sizes = TRUE)
liver_norm_counts <- t(liver_norm_counts)

In [16]:
# Load each CSV
young_liver <- read.csv(file.path("young_liver.csv"),
                        row.names = 1,
                        check.names = FALSE,
                        stringsAsFactors = FALSE)

young_brain_hypothalamus <- read.csv(file.path("young_brain_hypothalamus.csv"),
                                     row.names = 1,
                                     check.names = FALSE,
                                     stringsAsFactors = FALSE)

aged_liver <- read.csv(file.path("aged_liver.csv"),
                       row.names = 1,
                       check.names = FALSE,
                       stringsAsFactors = FALSE)

aged_brain_hypothalamus <- read.csv(file.path("aged_brain_hypothalamus.csv"),
                                    row.names = 1,
                                    check.names = FALSE,
                                    stringsAsFactors = FALSE)


In [17]:
young_liver_expr <- liver_norm_counts[rownames(liver_norm_counts) %in% rownames(young_liver), ]
young_brain_expr <- brain_hypothalamus_norm_counts[rownames(brain_hypothalamus_norm_counts) %in% rownames(young_brain_hypothalamus), ]
aged_liver_expr <- liver_norm_counts[rownames(liver_norm_counts) %in% rownames(aged_liver), ]
aged_brain_expr <- brain_hypothalamus_norm_counts[rownames(brain_hypothalamus_norm_counts) %in% rownames(aged_brain_hypothalamus), ]

# Save each matrix as a CSV
write.csv(young_liver_expr, file = "MultiCens_young_liver_expr.csv", row.names = TRUE)
write.csv(young_brain_expr, file = "MultiCens_young_brain_hypothalamus_expr.csv", row.names = TRUE)
write.csv(aged_liver_expr, file = "MultiCens_aged_liver_expr.csv", row.names = TRUE)
write.csv(aged_brain_expr, file = "MultiCens_aged_brain_hypothalamus_expr.csv", row.names = TRUE)